# 🧪 9 Model Comparison Test (Before/After Analysis)

**Purpose**: Compare Fine-tuned models vs Baseline (kor_unsmile)

| Category | Model |
|----------|-------|
| Baseline | smilegate-ai/kor_unsmile |
| LoRA v1 | Game KcELECTRA, Tutorial kcbert |
| LoRA v2 | Game KcELECTRA (+ Game Data), Tutorial kcbert (+ Game Data) |
| Full v1 | Game KcELECTRA, Tutorial kcbert |
| Full v2 | Game KcELECTRA (+ Game Data), Tutorial kcbert (+ Game Data) |

In [ ]:
# 1. Environment Setup
import os, torch, pandas as pd, numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import precision_recall_fscore_support, label_ranking_average_precision_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

# No Korean font needed - using English labels
plt.rcParams['axes.unicode_minus'] = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Configuration
MAX_LENGTH = 128
LABEL_NAMES = ["여성/가족", "남성", "성소수자", "인종/국적", "연령", "지역", "종교", "기타 혐오", "악플/욕설", "clean"]

# Base tokenizers
BASE_TOKENIZERS = {
    'kcelectra': 'beomi/KcELECTRA-base-v2022',
    'kcbert': 'beomi/kcbert-base',
    'unsmile': 'smilegate-ai/kor_unsmile'
}

# Model config: (model_path, tokenizer_key)
MODELS = {
    "Baseline (kor_unsmile)": ("smilegate-ai/kor_unsmile", "unsmile"),
    "Full v1 Game": ("../4_2_Full_Fine_Tuning/v1_corrected_only/output/full_game_kcelectra/best_model", "kcelectra"),
    "Full v1 Tutorial": ("../4_2_Full_Fine_Tuning/v1_corrected_only/output/full_tutorial_kcbert/best_model", "kcbert"),
    "Full v2 Game": ("../4_2_Full_Fine_Tuning/v2_corrected_plus_collected/output/full_game_kcelectra_v2/best_model", "kcelectra"),
    "Full v2 Tutorial": ("../4_2_Full_Fine_Tuning/v2_corrected_plus_collected/output/full_tutorial_kcbert_v2/best_model", "kcbert"),
    "LoRA v1 Game": ("../4_1_LoRA_Fine_Tuning/v1_corrected_only/output/lora_game_kcelectra/merged_model", "kcelectra"),
    "LoRA v1 Tutorial": ("../4_1_LoRA_Fine_Tuning/v1_corrected_only/output/lora_tutorial_kcbert/merged_model", "kcbert"),
    "LoRA v2 Game": ("../4_1_LoRA_Fine_Tuning/v2_corrected_plus_collected/output/lora_game_kcelectra_v2/merged_model", "kcelectra"),
    "LoRA v2 Tutorial": ("../4_1_LoRA_Fine_Tuning/v2_corrected_plus_collected/output/lora_tutorial_kcbert_v2/merged_model", "kcbert")
}

print(f"Models to test: {len(MODELS)}")
for name, (path, tok) in MODELS.items():
    print(f"   - {name} (tokenizer: {tok})")

In [ ]:
# 3. Load Test Data
test_df = pd.read_csv("../0_Data_Collection/datasets/test_set.tsv", sep='\t')

for col in LABEL_NAMES:
    test_df[col] = pd.to_numeric(test_df[col], errors='coerce').fillna(0).astype(int)

print(f"Test data: {len(test_df)} samples")
print(f"   - Abuse: {test_df['악플/욕설'].sum()} ({test_df['악플/욕설'].sum()/len(test_df)*100:.1f}%)")
print(f"   - Clean: {test_df['clean'].sum()} ({test_df['clean'].sum()/len(test_df)*100:.1f}%)")
test_df[['문장', '악플/욕설', 'clean']].head(10)

In [ ]:
# 4. Evaluation Function
def evaluate_model(model_name, model_path, tokenizer_key, test_df):
    print(f"\n{'='*60}")
    print(f"Evaluating: {model_name}")
    print(f"   Model: {model_path}")
    print(f"   Tokenizer: {BASE_TOKENIZERS[tokenizer_key]}")
    
    try:
        tokenizer = AutoTokenizer.from_pretrained(BASE_TOKENIZERS[tokenizer_key])
        model = AutoModelForSequenceClassification.from_pretrained(model_path).to(DEVICE)
        model.eval()
        
        all_preds, all_probs, all_labels = [], [], []
        
        with torch.no_grad():
            for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc=model_name):
                inputs = tokenizer(str(row['문장']), padding='max_length', truncation=True, 
                                   max_length=MAX_LENGTH, return_tensors='pt').to(DEVICE)
                outputs = model(**inputs)
                probs = torch.sigmoid(outputs.logits).cpu().numpy()[0]
                preds = (probs > 0.5).astype(int)
                all_probs.append(probs)
                all_preds.append(preds)
                all_labels.append([int(row[col]) for col in LABEL_NAMES])
        
        all_probs, all_preds, all_labels = np.array(all_probs), np.array(all_preds), np.array(all_labels)
        
        lrap = label_ranking_average_precision_score(all_labels, all_probs)
        abuse_p, abuse_r, abuse_f1, _ = precision_recall_fscore_support(all_labels[:,8], all_preds[:,8], average='binary', zero_division=0)
        clean_p, clean_r, clean_f1, _ = precision_recall_fscore_support(all_labels[:,9], all_preds[:,9], average='binary', zero_division=0)
        abuse_cm = confusion_matrix(all_labels[:,8], all_preds[:,8])
        clean_cm = confusion_matrix(all_labels[:,9], all_preds[:,9])
        
        print(f"   LRAP: {lrap:.4f} | Abuse R: {abuse_r:.4f}, F1: {abuse_f1:.4f}")
        
        del model, tokenizer
        torch.cuda.empty_cache()
        
        return {'model': model_name, 'lrap': round(lrap,4), 'abuse_precision': round(abuse_p,4),
                'abuse_recall': round(abuse_r,4), 'abuse_f1': round(abuse_f1,4),
                'clean_precision': round(clean_p,4), 'clean_recall': round(clean_r,4), 'clean_f1': round(clean_f1,4),
                'abuse_cm': abuse_cm, 'clean_cm': clean_cm, 'predictions': all_preds, 'probabilities': all_probs}
    except Exception as e:
        print(f"   Error: {e}")
        return {'model': model_name, 'error': str(e)}

In [ ]:
# 5. Evaluate All Models
results = []
detailed_results = {}
for model_name, (model_path, tokenizer_key) in MODELS.items():
    result = evaluate_model(model_name, model_path, tokenizer_key, test_df)
    results.append(result)
    detailed_results[model_name] = result
print("\n" + "="*60 + "\nAll models evaluated!")

In [ ]:
# 6. Results DataFrame
cols = ['model', 'lrap', 'abuse_precision', 'abuse_recall', 'abuse_f1', 'clean_precision', 'clean_recall', 'clean_f1']
results_df = pd.DataFrame([{k: r.get(k) for k in cols} for r in results if 'error' not in r])
results_df = results_df.sort_values('abuse_recall', ascending=False).reset_index(drop=True)
print("All Results (Sorted by Abuse Recall)")
results_df

In [ ]:
# 7. Improvement vs Baseline
baseline = results_df[results_df['model'].str.contains('Baseline')].iloc[0]
improvement_df = results_df.copy()
improvement_df['abuse_r_diff'] = (improvement_df['abuse_recall'] - baseline['abuse_recall']).round(4)
improvement_df['abuse_r_pct'] = ((improvement_df['abuse_recall'] - baseline['abuse_recall']) / baseline['abuse_recall'] * 100).round(1)
improvement_df['abuse_f1_diff'] = (improvement_df['abuse_f1'] - baseline['abuse_f1']).round(4)
improvement_df['lrap_diff'] = (improvement_df['lrap'] - baseline['lrap']).round(4)
print(f"Baseline: Abuse Recall {baseline['abuse_recall']:.4f} | F1 {baseline['abuse_f1']:.4f} | LRAP {baseline['lrap']:.4f}")
improvement_df[['model', 'abuse_recall', 'abuse_r_diff', 'abuse_r_pct', 'abuse_f1_diff', 'lrap_diff']]

In [ ]:
# 8. Visualization - Before/After Charts (ENGLISH LABELS)
os.makedirs('./results', exist_ok=True)
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
colors = ['#FF6B6B' if 'Baseline' in m else '#4ECDC4' if 'v2' in m else '#45B7D1' for m in results_df['model']]

# 1) Abuse Recall
ax1 = axes[0, 0]
bars = ax1.barh(results_df['model'], results_df['abuse_recall'], color=colors)
ax1.axvline(x=baseline['abuse_recall'], color='red', linestyle='--', linewidth=2, label=f"Baseline: {baseline['abuse_recall']:.4f}")
ax1.set_xlabel('Abuse Recall'); ax1.set_title('Abuse Recall Comparison', fontweight='bold'); ax1.legend()
for bar, val in zip(bars, results_df['abuse_recall']): ax1.text(val+0.01, bar.get_y()+bar.get_height()/2, f'{val:.4f}', va='center')

# 2) Abuse F1
ax2 = axes[0, 1]
bars = ax2.barh(results_df['model'], results_df['abuse_f1'], color=colors)
ax2.axvline(x=baseline['abuse_f1'], color='red', linestyle='--', linewidth=2, label=f"Baseline: {baseline['abuse_f1']:.4f}")
ax2.set_xlabel('Abuse F1'); ax2.set_title('Abuse F1 Comparison', fontweight='bold'); ax2.legend()
for bar, val in zip(bars, results_df['abuse_f1']): ax2.text(val+0.01, bar.get_y()+bar.get_height()/2, f'{val:.4f}', va='center')

# 3) LRAP
ax3 = axes[1, 0]
bars = ax3.barh(results_df['model'], results_df['lrap'], color=colors)
ax3.axvline(x=baseline['lrap'], color='red', linestyle='--', linewidth=2, label=f"Baseline: {baseline['lrap']:.4f}")
ax3.set_xlabel('LRAP'); ax3.set_title('LRAP Comparison', fontweight='bold'); ax3.legend()
for bar, val in zip(bars, results_df['lrap']): ax3.text(val+0.005, bar.get_y()+bar.get_height()/2, f'{val:.4f}', va='center')

# 4) Improvement %
ax4 = axes[1, 1]
imp_df = improvement_df[~improvement_df['model'].str.contains('Baseline')].copy()
imp_colors = ['#2ECC71' if v > 0 else '#E74C3C' for v in imp_df['abuse_r_pct']]
bars = ax4.barh(imp_df['model'], imp_df['abuse_r_pct'], color=imp_colors)
ax4.axvline(x=0, color='black', linestyle='-', linewidth=1)
ax4.set_xlabel('Improvement (%)'); ax4.set_title('Abuse Recall Improvement vs Baseline', fontweight='bold')
for bar, val in zip(bars, imp_df['abuse_r_pct']): ax4.text(val+(1 if val>=0 else -3), bar.get_y()+bar.get_height()/2, f'{val:+.1f}%', va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('./results/before_after_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 9. Heatmap (ENGLISH LABELS)
fig, ax = plt.subplots(figsize=(12, 8))
heatmap_df = results_df.set_index('model')[['lrap', 'abuse_precision', 'abuse_recall', 'abuse_f1', 'clean_precision', 'clean_recall', 'clean_f1']]
sns.heatmap(heatmap_df, annot=True, fmt='.4f', cmap='RdYlGn', ax=ax, linewidths=0.5, cbar_kws={'label': 'Score'})
ax.set_title('All Metrics Heatmap', fontweight='bold')
plt.tight_layout()
plt.savefig('./results/metrics_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 10. LoRA vs Full (ENGLISH LABELS)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
lora_models = results_df[results_df['model'].str.contains('LoRA')]
full_models = results_df[results_df['model'].str.contains('Full')]
width = 0.35

ax1 = axes[0]
x = range(len(lora_models))
ax1.bar([i-width/2 for i in x], lora_models['abuse_recall'], width, label='Abuse Recall', color='#3498DB')
ax1.bar([i+width/2 for i in x], lora_models['abuse_f1'], width, label='Abuse F1', color='#E74C3C')
ax1.axhline(y=baseline['abuse_recall'], color='blue', linestyle='--', alpha=0.5)
ax1.set_xticks(x); ax1.set_xticklabels([m.replace('LoRA ','') for m in lora_models['model']], rotation=15)
ax1.set_ylabel('Score'); ax1.set_title('LoRA Fine-tuning Results', fontweight='bold'); ax1.legend(); ax1.set_ylim(0,1)

ax2 = axes[1]
x = range(len(full_models))
ax2.bar([i-width/2 for i in x], full_models['abuse_recall'], width, label='Abuse Recall', color='#3498DB')
ax2.bar([i+width/2 for i in x], full_models['abuse_f1'], width, label='Abuse F1', color='#E74C3C')
ax2.axhline(y=baseline['abuse_recall'], color='blue', linestyle='--', alpha=0.5)
ax2.set_xticks(x); ax2.set_xticklabels([m.replace('Full ','') for m in full_models['model']], rotation=15)
ax2.set_ylabel('Score'); ax2.set_title('Full Fine-tuning Results', fontweight='bold'); ax2.legend(); ax2.set_ylim(0,1)

plt.tight_layout()
plt.savefig('./results/lora_vs_full.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 11. v1 vs v2 (ENGLISH LABELS)
fig, ax = plt.subplots(figsize=(12, 6))
comparison = []
for method in ['LoRA', 'Full']:
    for base in ['Game', 'Tutorial']:
        v1_row = results_df[results_df['model'] == f"{method} v1 {base}"]
        v2_row = results_df[results_df['model'] == f"{method} v2 {base}"]
        if len(v1_row) > 0 and len(v2_row) > 0:
            comparison.append({'model': f"{method} {base}", 'v1_recall': v1_row['abuse_recall'].values[0],
                               'v2_recall': v2_row['abuse_recall'].values[0],
                               'diff': v2_row['abuse_recall'].values[0] - v1_row['abuse_recall'].values[0]})
comp_df = pd.DataFrame(comparison)
x = range(len(comp_df)); width = 0.35
ax.bar([i-width/2 for i in x], comp_df['v1_recall'], width, label='v1 (UnSmile Only)', color='#95A5A6')
ax.bar([i+width/2 for i in x], comp_df['v2_recall'], width, label='v2 (+ Game Data)', color='#27AE60')
ax.axhline(y=baseline['abuse_recall'], color='red', linestyle='--', linewidth=2, label=f"Baseline: {baseline['abuse_recall']:.4f}")
ax.set_xticks(x); ax.set_xticklabels(comp_df['model'])
ax.set_ylabel('Abuse Recall'); ax.set_title('Game Data Effect (v1 vs v2)', fontweight='bold'); ax.legend(); ax.set_ylim(0,1)
for i, row in comp_df.iterrows():
    ax.annotate(f"{row['diff']:+.4f}", xy=(i, max(row['v1_recall'], row['v2_recall'])+0.02), ha='center', fontweight='bold', color='#27AE60' if row['diff']>0 else '#E74C3C')
plt.tight_layout()
plt.savefig('./results/v1_vs_v2_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 12. Save CSV Results
results_df.to_csv('./results/game_test_results.csv', index=False)
improvement_df.to_csv('./results/improvement_analysis.csv', index=False)
print("Saved: game_test_results.csv, improvement_analysis.csv")

In [ ]:
# 13. Generate Korean README (results folder)
best_model = results_df.iloc[0]
best_imp = improvement_df[~improvement_df['model'].str.contains('Baseline')].sort_values('abuse_r_pct', ascending=False).iloc[0]
v2_avg = results_df[results_df['model'].str.contains('v2')]['abuse_recall'].mean()
v1_avg = results_df[results_df['model'].str.contains('v1')]['abuse_recall'].mean()
lora_avg = results_df[results_df['model'].str.contains('LoRA')]['abuse_recall'].mean()
full_avg = results_df[results_df['model'].str.contains('Full')]['abuse_recall'].mean()

readme = f'''# 모델 비교 분석 결과

## 📋 개요
본 분석은 혐오 표현 탐지를 위한 **9개 모델**의 성능을 비교 평가한 결과입니다.
게임 내 음성 채팅(STT) 데이터를 활용하여 욕설/악플 탐지 성능을 측정하였습니다.

- **테스트 데이터**: game_test.tsv ({len(test_df)}건)
- **핵심 평가 지표**: Abuse Recall (욕설 탐지율)
- **분류 임계값**: 0.5

---

## 🏆 최종 결과

### 최고 성능 모델: `{best_model["model"]}`
| 지표 | 값 | Baseline 대비 |
|------|-----|---------------|
| **Abuse Recall** | {best_model["abuse_recall"]:.4f} | +{(best_model["abuse_recall"]-baseline["abuse_recall"])*100:.1f}%p |
| **Abuse F1** | {best_model["abuse_f1"]:.4f} | +{(best_model["abuse_f1"]-baseline["abuse_f1"])*100:.1f}%p |
| **LRAP** | {best_model["lrap"]:.4f} | +{(best_model["lrap"]-baseline["lrap"])*100:.1f}%p |

### Baseline 모델: `{baseline["model"]}`
| 지표 | 값 |
|------|-----|
| Abuse Recall | {baseline["abuse_recall"]:.4f} |
| Abuse F1 | {baseline["abuse_f1"]:.4f} |
| LRAP | {baseline["lrap"]:.4f} |

---

## 📊 전체 모델 순위

| 순위 | 모델명 | Abuse Recall | Abuse F1 | LRAP | 개선율 |
|------|--------|--------------|----------|------|--------|
'''
for i, (_, row) in enumerate(results_df.iterrows(), 1):
    imp_row = improvement_df[improvement_df['model'] == row['model']].iloc[0]
    pct = f"+{imp_row['abuse_r_pct']:.1f}%" if 'Baseline' not in row['model'] else '-'
    readme += f"| {i} | {row['model']} | {row['abuse_recall']:.4f} | {row['abuse_f1']:.4f} | {row['lrap']:.4f} | {pct} |\n"

readme += f'''
---

## 🔍 분석 결과

### 1. 게임 데이터 추가 효과 (v1 vs v2)
게임 특화 데이터를 추가한 **v2 모델**이 v1 대비 일관된 성능 향상을 보였습니다.

- v1 평균 Abuse Recall: **{v1_avg:.4f}**
- v2 평균 Abuse Recall: **{v2_avg:.4f}**
- 평균 개선: **+{(v2_avg - v1_avg)*100:.1f}%p**

**결론**: 도메인 특화 데이터 추가는 모델 성능 향상에 매우 효과적입니다.

### 2. Fine-tuning 방식 비교 (LoRA vs Full)
- LoRA 평균 Abuse Recall: **{lora_avg:.4f}**
- Full FT 평균 Abuse Recall: **{full_avg:.4f}**

**결론**: 두 방식 모두 유사한 성능을 달성합니다. LoRA는 학습 효율성(메모리, 속도)이 장점입니다.

---

## 📈 시각화 자료 설명

### 1. before_after_comparison.png
**4개 패널로 구성된 Before/After 비교 차트**

- **좌상단 (Abuse Recall)**: 모든 모델의 욕설 탐지율 비교. 빨간 점선은 Baseline 기준선.
- **우상단 (Abuse F1)**: Precision과 Recall의 조화 평균. 균형 잡힌 성능 지표.
- **좌하단 (LRAP)**: Label Ranking Average Precision. 다중 라벨 분류 성능.
- **우하단 (Improvement %)**: Baseline 대비 개선율. 녹색=개선, 빨간색=하락.

**해석**: v2 모델들이 상위권에 위치하며, 게임 데이터 추가의 효과가 명확히 나타납니다.

### 2. metrics_heatmap.png
**전체 메트릭 히트맵**

모든 모델 × 모든 평가 지표를 한눈에 비교할 수 있는 히트맵입니다.
- 녹색이 진할수록 높은 성능
- 빨간색이 진할수록 낮은 성능

**해석**: 최상위 모델들은 대부분의 지표에서 녹색 계열을 보입니다.

### 3. lora_vs_full.png
**LoRA vs Full Fine-tuning 비교**

- 좌측: LoRA 방식으로 학습된 4개 모델
- 우측: Full Fine-tuning으로 학습된 4개 모델
- 파란색 막대: Abuse Recall
- 빨간색 막대: Abuse F1
- 점선: Baseline 기준

**해석**: 두 방식 모두 Baseline을 크게 상회하며, v2 모델들이 더 높은 성능을 보입니다.

### 4. v1_vs_v2_comparison.png
**게임 데이터 추가 효과 비교**

- 회색: v1 (UnSmile 데이터만 사용)
- 녹색: v2 (UnSmile + 게임 데이터)
- 막대 위 숫자: v2 - v1 차이값

**해석**: 모든 조합에서 v2가 v1보다 높은 성능을 보여, 게임 특화 데이터의 중요성을 입증합니다.

---

## 📁 산출물 목록

| 파일명 | 설명 |
|--------|------|
| `game_test_results.csv` | 전체 모델별 상세 메트릭 결과 |
| `improvement_analysis.csv` | Baseline 대비 개선율 분석 |
| `before_after_comparison.png` | 4개 주요 지표 비교 차트 |
| `metrics_heatmap.png` | 전체 메트릭 히트맵 |
| `lora_vs_full.png` | LoRA vs Full FT 비교 차트 |
| `v1_vs_v2_comparison.png` | 게임 데이터 효과 비교 차트 |

---

## 💡 권장 사항

1. **프로덕션 배포**: `{best_model["model"]}` 모델 권장
2. **리소스 제약 시**: LoRA 모델 선택 (유사 성능, 높은 효율)
3. **추가 개선**: 더 많은 게임 도메인 데이터 수집 권장

---
*생성일시: {pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")}*
'''

with open('./results/README.md', 'w', encoding='utf-8') as f:
    f.write(readme)
print("✅ 한글 README 생성 완료: ./results/README.md")

In [ ]:
# 14. Final Summary
print("="*60)
print("FINAL SUMMARY")
print("="*60)
print(f"\nBest Model: {best_model['model']}")
print(f"   Abuse Recall: {best_model['abuse_recall']:.4f} (+{best_imp['abuse_r_pct']:.1f}% vs Baseline)")
print(f"\nAll outputs saved to ./results/")